# GOL-10 — FRACTRAN : calculer avec des fractions

**Navigation** : [GOL-09 — Perplexité structurelle Hashlife](GOL-09-Perplexite-Structurelle-Hashlife.ipynb) · fin de la série GOL · [Index](../README.md) · le versant formalisé en Lean : [Lean-16b — FRACTRAN kernel natif](../Lean-16b-Conway-FRACTRAN-Lean-Native.ipynb)

## Objectifs d'apprentissage

- Exécuter un programme FRACTRAN : une liste ordonnée de fractions et un entier.
- Lire l'état comme des registres encodés dans les exposants d'une factorisation première.
- Vérifier quatre programmes exacts de Conway : addition, multiplication, Fibonacci et Collatz.
- Comprendre pourquoi ce modèle est universel par la correspondance avec les machines de Minsky.

## Prérequis

- Décomposition en facteurs premiers et lois des exposants.
- Notions de suites récurrentes et de suite de Collatz.
- Bases de Python 3 (boucles, fonctions, tuples).

## Durée estimée

Environ 45 minutes.

## Source primaire

- Reg Braithwaite (2020), *Remembering John Conway's FRACTRAN, a ridiculous, yet surprisingly deep language*. Copie archivée : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Logic Programming and Neuro-Symbolic AI\2020 - Braithwaite - Remembering John Conways FRACTRAN.pdf`.
- John H. Conway (1987), « FRACTRAN: A Simple Universal Programming Language for Arithmetic », dans *Open Problems in Communication and Computation*, Springer.

Tous les programmes exécutés ici sont repris texte source à l'appui depuis ces deux références.

## 1. Le modèle FRACTRAN

Un programme FRACTRAN est une **liste ordonnée de fractions positives**. L'état du calcul est un entier strictement positif $n$.

À chaque étape :

1. on parcourt la liste de gauche à droite ;
2. on cherche la **première** fraction $f$ telle que $n \times f$ soit un entier ;
3. on remplace $n$ par $n \times f$ et on recommence ;
4. si aucune fraction ne convient, le programme s'arrête.

Il n'y a ni variable, ni test explicite, ni boucle : seulement un entier et des fractions.

### Exemple guidé — l'interpréteur

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from fractions import Fraction
from math import prod
from typing import Iterable


@dataclass(frozen=True)
class FractranResult:
    """Résultat d'une exécution FRACTRAN."""

    states: list[int]
    rule_indices: list[int]
    halted: bool

    @property
    def steps(self) -> int:
        return len(self.states) - 1


def parse_fractran(program_text: str) -> tuple[Fraction, ...]:
    """Convertit une liste textuelle de fractions en tuple de Fraction."""
    return tuple(Fraction(item.strip()) for item in program_text.split(",") if item.strip())


def run_fractran(
    fractions: Iterable[Fraction],
    initial_state: int,
    max_steps: int = 100_000,
) -> FractranResult:
    """Exécute un programme FRACTRAN jusqu'à l'arrêt ou max_steps."""
    fractions = tuple(fractions)
    state = int(initial_state)
    states = [state]
    rule_indices: list[int] = []

    for _ in range(max_steps):
        for index, fraction in enumerate(fractions):
            if state % fraction.denominator == 0:
                state = state * fraction.numerator // fraction.denominator
                states.append(state)
                rule_indices.append(index)
                break
        else:
            return FractranResult(states, rule_indices, halted=True)

    return FractranResult(states, rule_indices, halted=False)


def exact_log_base(value: int, base: int) -> int | None:
    """Retourne k si value == base**k, sinon None."""
    if value <= 0:
        return None
    if base == 2:
        # Un entier positif est une puissance de 2 ssi un seul bit vaut 1.
        if value & (value - 1):
            return None
        return value.bit_length() - 1
    exponent = 0
    while value > 1 and value % base == 0:
        value //= base
        exponent += 1
    return exponent if value == 1 else None


def format_registers(state: int, primes: Iterable[int]) -> str:
    """Affiche les exposants connus d'un état FRACTRAN."""
    parts: list[str] = []
    for prime in primes:
        exponent = 0
        working = state
        while working % prime == 0:
            working //= prime
            exponent += 1
        if exponent:
            parts.append(f"{prime}^{exponent}")
    return " * ".join(parts) if parts else str(state)


print("Interpréteur FRACTRAN prêt.")
print(f"Première fraction applicable à 2 dans (3/2,) : {run_fractran((Fraction(3, 2),), 2).states}")
print(f"Première fraction applicable à 5 dans (3/2,) : {run_fractran((Fraction(3, 2),), 5).states}")

Interpréteur FRACTRAN prêt.
Première fraction applicable à 2 dans (3/2,) : [2, 3]
Première fraction applicable à 5 dans (3/2,) : [5]


### Lecture du résultat

L'interpréteur tient en une boucle : la condition `state % fraction.denominator == 0` est exactement le test « $n \times f$ est-il un entier ? ». Sur l'exemple, $2$ devient $3$ en une étape, tandis que $5$ est déjà un état final car $5 \times \frac{3}{2}$ n'est pas entier.

## 2. L'addition : deux fractions suffisent

Le programme d'addition de la source est :

$$\frac{2}{3},\ \frac{2}{5}$$

L'entrée est $3^a \cdot 5^b$ : le registre $a$ vit dans l'exposant de 3, le registre $b$ dans celui de 5. Chaque fraction consomme une unité d'un registre d'entrée et produit une unité du registre de sortie, l'exposant de 2. À l'arrêt, l'état est exactement $2^{a+b}$.

### Exemple guidé — additionner 2 et 3

In [2]:
addition_program = parse_fractran("2/3, 2/5")
a, b = 2, 3
initial_state = 3**a * 5**b
addition_result = run_fractran(addition_program, initial_state)

print(f"Entrée : 3^{a} * 5^{b} = {initial_state}")
for step, (state, rule_index) in enumerate(zip(addition_result.states, [None, *addition_result.rule_indices])):
    fraction = "état initial" if rule_index is None else str(addition_program[rule_index])
    print(f"{step:>2} | {state:>5} | {fraction:<12} | {format_registers(state, (2, 3, 5))}")

decoded_sum = exact_log_base(addition_result.states[-1], 2)
print(f"Arrêt : {addition_result.halted}, étapes : {addition_result.steps}, état final : {addition_result.states[-1]}")
print(f"2^{decoded_sum} = {addition_result.states[-1]}, donc a + b = {decoded_sum}")
assert addition_result.halted
assert decoded_sum == a + b

Entrée : 3^2 * 5^3 = 1125
 0 |  1125 | état initial | 3^2 * 5^3
 1 |   750 | 2/3          | 2^1 * 3^1 * 5^3
 2 |   500 | 2/3          | 2^2 * 5^3
 3 |   200 | 2/5          | 2^3 * 5^2
 4 |    80 | 2/5          | 2^4 * 5^1
 5 |    32 | 2/5          | 2^5
Arrêt : True, étapes : 5, état final : 32
2^5 = 32, donc a + b = 5


### Lecture du résultat

L'état initial $1125 = 3^2 \cdot 5^3$ devient $32 = 2^5$ en cinq étapes. La trace montre le mécanisme fondamental de FRACTRAN : **les exposants des nombres premiers sont les registres de la machine**. Multiplier par $\frac{2}{3}$ décrémente le registre 3 et incrémente le registre 2.

### Exercice 1 — doubler un registre

**Objectif** : écrire un programme FRACTRAN à une fraction qui transforme $3^a$ en $2^{2a}$.

`# Indice` : le numérateur peut contenir plus d'un facteur 2.

La cellule doit rester exécutable même si l'exercice n'est pas complété.

In [3]:
doubling_program = None  # TODO etudiant : remplacer par un tuple contenant une seule Fraction

if doubling_program is None:
    print("Exercice a completer")
else:
    exercise_state = 3**4
    exercise_result = run_fractran(doubling_program, exercise_state)
    exercise_value = exact_log_base(exercise_result.states[-1], 2)
    print(f"État final : {exercise_result.states[-1]}, exposant de 2 : {exercise_value}")

Exercice a completer


## 3. La multiplication : huit fractions compilées d'une machine de Minsky

La source donne ce programme multiplicateur :

$$\frac{182}{55},\ \frac{17}{11},\ \frac{95}{119},\ \frac{1}{17},\ \frac{11}{13},\ \frac{17}{19},\ \frac{11}{3},\ \frac{1}{5}$$

L'entrée est $3^a \cdot 5^b$ et la sortie est $2^{ab}$. Ce programme n'est pas écrit à la main comme une astuce : c'est la **compilation d'une machine de Minsky à un état**. Les registres y sont indexés par les nombres premiers ($1 \mapsto 2$, $2 \mapsto 3$, $3 \mapsto 5$, $4 \mapsto 7$, $5 \mapsto 11$, …) : les registres 1 à 4, soit les exposants de 2, 3, 5 et 7, portent les **données** — produit, compteur, multiplicande et temporaire — et les quatre registres suivants, 5 à 8, soit les exposants de **11, 13, 17 et 19**, servent de **drapeaux de contrôle**. Nous verrons cette compilation à la section 6.

### Exemple guidé — trois multiplications vérifiées

In [4]:
multiplication_program = parse_fractran("182/55, 17/11, 95/119, 1/17, 11/13, 17/19, 11/3, 1/5")

print("  a  b | étapes | état final           | attendu")
for a, b in [(2, 2), (3, 13), (4, 5)]:
    multiplication_result = run_fractran(multiplication_program, 3**a * 5**b)
    expected = 2 ** (a * b)
    print(f"{a:>3} {b:>3} | {multiplication_result.steps:>6} | {multiplication_result.states[-1]:>20} | {expected}")
    assert multiplication_result.halted
    assert multiplication_result.states[-1] == expected

  a  b | étapes | état final           | attendu
  2   2 |     24 |                   16 | 16
  3  13 |    178 |         549755813888 | 549755813888
  4   5 |     97 |              1048576 | 1048576


### Lecture du résultat

Les trois cas s'arrêtent exactement sur $2^{ab}$ : $16$ pour $2 \times 2$ en 24 étapes, $549755813888 = 2^{39}$ pour $3 \times 13$ en 178 étapes, et $1048576 = 2^{20}$ pour $4 \times 5$ en 97 étapes. Le nombre d'étapes n'est pas $ab$ : les drapeaux de contrôle ajoutent des transitions internes, mais l'arithmétique des exposants garantit le résultat.

### Exercice 2 — additionner trois registres

**Objectif** : étendre l'additionneur à trois entrées. Le programme doit transformer $3^a \cdot 5^b \cdot 7^c$ en $2^{a+b+c}$.

`# Indice` : une fraction par registre d'entrée, toutes vers le registre 2.
`# Etape 1` : recopier le programme d'addition.
`# Etape 2` : ajouter la fraction qui consomme le registre 7.

In [5]:
triple_addition_program = None  # TODO etudiant : tuple de trois Fraction

if triple_addition_program is None:
    print("Exercice a completer")
else:
    exercise_result = run_fractran(triple_addition_program, 3**2 * 5**4 * 7**3)
    decoded = exact_log_base(exercise_result.states[-1], 2)
    print(f"Arrêt : {exercise_result.halted}, exposant final de 2 : {decoded}")

Exercice a completer


## 4. Fibonacci par Conway

Le programme de Fibonacci donné par Conway comporte **15 fractions** :

$$\frac{17}{65},\ \frac{133}{34},\ \frac{17}{19},\ \frac{23}{17},\ \frac{2233}{69},\ \frac{23}{29},\ \frac{31}{23},\ \frac{74}{341},\ \frac{31}{37},\ \frac{41}{31},\ \frac{129}{287},\ \frac{41}{43},\ \frac{13}{41},\ \frac{1}{13},\ \frac{1}{3}$$

Pour calculer $\operatorname{fib}(x)$, on démarre de :

$$n = 78 \cdot 5^{x-1}$$

et, à l'arrêt, l'état final est une puissance exacte de 2 dont l'exposant est la réponse.

### Exemple guidé — fib(1) à fib(10)

In [6]:
fibonacci_program = parse_fractran(
    "17/65, 133/34, 17/19, 23/17, 2233/69, 23/29, 31/23, "
    "74/341, 31/37, 41/31, 129/287, 41/43, 13/41, 1/13, 1/3"
)

def fibonacci_reference(count: int) -> list[int]:
    """Suite de Fibonacci de référence : fib(1) = fib(2) = 1."""
    previous, current = 0, 1
    sequence: list[int] = []
    for _ in range(count):
        sequence.append(current)
        previous, current = current, previous + current
    return sequence


reference = fibonacci_reference(10)
print("  x | fib(x) | étapes | état final")
for x in range(1, 11):
    fibonacci_result = run_fractran(fibonacci_program, 78 * 5 ** (x - 1), max_steps=10_000)
    decoded = exact_log_base(fibonacci_result.states[-1], 2)
    print(f"{x:>3} | {decoded:>6} | {fibonacci_result.steps:>6} | {fibonacci_result.states[-1]}")
    assert fibonacci_result.halted
    assert decoded == reference[x - 1]

  x | fib(x) | étapes | état final
  1 |      1 |      2 | 2
  2 |      1 |     18 | 2
  3 |      2 |     40 | 4
  4 |      3 |     73 | 8
  5 |      5 |    123 | 32
  6 |      8 |    201 | 256
  7 |     13 |    324 | 8192
  8 |     21 |    520 | 2097152
  9 |     34 |    834 | 17179869184
 10 |     55 |   1339 | 36028797018963968


### Lecture du résultat

Les dix valeurs correspondent exactement à la suite $1, 1, 2, 3, 5, 8, 13, 21, 34, 55$. Le nombre d'étapes monte de 2 pour $x=1$ à 1339 pour $x=10$, tandis que l'état final monte jusqu'à $2^{55}$ : la même liste de 15 fractions encode une itération complète de la récurrence, sans aucune autre structure de contrôle.

## 5. Collatz : un programme dont la terminaison est un problème ouvert

La source donne ce programme Collatz à **12 fractions** :

$$\frac{165}{14},\ \frac{11}{63},\ \frac{38}{21},\ \frac{13}{7},\ \frac{34}{325},\ \frac{1}{13},\ \frac{184}{95},\ \frac{1}{19},\ \frac{7}{11},\ \frac{13}{17},\ \frac{19}{23},\ \frac{1575}{4}$$

On démarre de $2^x$. Chaque état de la forme $2^y$ donne le terme suivant de la suite de Collatz, et le programme s'arrête sur $2^1$ lorsque la suite atteint 1.

### Exemple guidé — suite extraite des puissances de 2

In [7]:
collatz_program = parse_fractran(
    "165/14, 11/63, 38/21, 13/7, 34/325, 1/13, "
    "184/95, 1/19, 7/11, 13/17, 19/23, 1575/4"
)

def extract_powers_of_two(states: list[int]) -> list[int]:
    """Extrait les exposants des états qui sont des puissances exactes de 2."""
    extracted: list[int] = []
    for state in states:
        exponent = exact_log_base(state, 2)
        if exponent is not None:
            extracted.append(exponent)
    return extracted


for x in (3, 6, 7):
    collatz_result = run_fractran(collatz_program, 2**x, max_steps=100_000)
    sequence = extract_powers_of_two(collatz_result.states)
    print(f"x={x} : étapes={collatz_result.steps}, suite={sequence}")
    assert collatz_result.halted

collatz_27 = run_fractran(collatz_program, 2**27, max_steps=600_000)
sequence_27 = extract_powers_of_two(collatz_27.states)
print(f"x=27 : étapes={collatz_27.steps}, termes={len(sequence_27)}, maximum={max(sequence_27)}, dernier={sequence_27[-1]}")
assert collatz_27.halted

x=3 : étapes=191, suite=[3, 10, 5, 16, 8, 4, 2, 1]
x=6 : étapes=214, suite=[6, 3, 10, 5, 16, 8, 4, 2, 1]
x=7 : étapes=1180, suite=[7, 22, 11, 34, 17, 52, 26, 13, 40, 20, 10, 5, 16, 8, 4, 2, 1]


x=27 : étapes=425865, termes=112, maximum=9232, dernier=1


### Lecture du résultat

Pour $x=3$, les états puissance de 2 redonnent la suite $3, 10, 5, 16, 8, 4, 2, 1$. Pour $x=27$, le programme FRACTRAN exécute 425865 transitions et retrouve une suite de 112 termes, avec un maximum de 9232 avant la descente vers 1. L'équivalence avec la conjecture de Collatz ne vaut que pour la **famille d'entrées $2^x$** : c'est sur ces états que chaque puissance de 2 redonne exactement le terme suivant de la suite, et le programme s'arrête sur $2^1$ si et seulement si la suite atteint 1. Sur un entier de départ quelconque, l'arrêt n'a plus rien à voir avec Collatz : la plupart des entrées n'appliquent aucune fraction et s'arrêtent en zéro étape — c'est le cas de **467** des 1 000 premiers entiers, dont 5, 9 et 27.

### Exercice 3 — suite de Collatz de référence

**Objectif** : écrire la suite de Collatz usuelle pour $x=9$, puis la comparer à l'extraction du programme FRACTRAN.

`# Indice` : si le terme courant est pair, le diviser par 2 ; sinon le remplacer par 3n + 1.
`# Etape 1` : construire la liste jusqu'à atteindre 1.
`# Etape 2` : appeler extract_powers_of_two sur run_fractran(collatz_program, 2**9).

In [8]:
collatz_nine_reference = None  # TODO etudiant : liste des termes de la suite pour x=9

if collatz_nine_reference is None:
    print("Exercice a completer")
else:
    fractran_nine = run_fractran(collatz_program, 2**9, max_steps=100_000)
    fractran_sequence = extract_powers_of_two(fractran_nine.states)
    print(f"Référence : {collatz_nine_reference}")
    print(f"FRACTRAN : {fractran_sequence}")

Exercice a completer


## 6. Pourquoi FRACTRAN est universel

Une **machine de Minsky** possède des registres contenant des entiers naturels et des règles qui incrémentent ou décrémentent ces registres, avec branchements conditionnels. Ces machines sont Turing-complètes.

La correspondance avec FRACTRAN se construit en trois étapes, détaillées dans la source :

1. les états d'une machine multi-états peuvent être encodés dans des **registres-drapeaux** supplémentaires, donnant une machine à un état ;
2. chaque registre $r_i$ est associé au $i$-ième nombre premier $p_i$, et sa valeur est l'exposant de $p_i$ dans $n$ ;
3. une règle devient une fraction : ses increments forment le numérateur, ses decrements le dénominateur.

| Invention de Conway | Modèle | Universalité |
|---|---|---|
| Jeu de la Vie (1970) | Automate cellulaire 2D | Turing-complet via les constructions de Gosper |
| FRACTRAN (1987) | Liste ordonnée de fractions | Turing-complet via les machines de Minsky |

Conway pousse le résultat jusqu'au programme **POLYGAME**, universel : toute fonction récursivement énumérable possède un numéro de catalogue $c$ tel que, partant de $c \cdot 2^{2^n}$, POLYGAME produit $2^{2^{f(n)}}$.

### Exemple guidé — compiler deux machines de Minsky

In [9]:
PRIMES = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29)

def compile_rule(action: dict[int, int], guard: dict[int, int]) -> Fraction:
    """Compile une règle de Minsky en fraction FRACTRAN.

    Le registre r est associé au prime PRIMES[r - 1]. Les increments
    forment le numérateur et les decrements le dénominateur.
    """
    numerator = prod(PRIMES[register - 1] ** amount for register, amount in action.items())
    denominator = prod(PRIMES[register - 1] ** amount for register, amount in guard.items())
    return Fraction(numerator, denominator)


# Addition : incrémenter le registre 1 en consommant le registre 2 ou 3.
minsky_addition = [
    ({1: 1}, {2: 1}),
    ({1: 1}, {3: 1}),
]

# Multiplicateur à un état de la source, drapeaux compris.
minsky_multiplication = [
    ({1: 1, 4: 1, 6: 1}, {3: 1, 5: 1}),
    ({7: 1}, {5: 1}),
    ({3: 1, 8: 1}, {4: 1, 7: 1}),
    ({1: 0}, {7: 1}),
    ({5: 1}, {6: 1}),
    ({7: 1}, {8: 1}),
    ({5: 1}, {2: 1}),
    ({1: 0}, {3: 1}),
]

compiled_addition = tuple(compile_rule(action, guard) for action, guard in minsky_addition)
compiled_multiplication = tuple(compile_rule(action, guard) for action, guard in minsky_multiplication)

print(f"Addition compilée : {', '.join(str(f) for f in compiled_addition)}")
print(f"Multiplication compilée : {', '.join(str(f) for f in compiled_multiplication)}")
print(f"Addition identique à la source : {compiled_addition == addition_program}")
print(f"Multiplication identique à la source : {compiled_multiplication == multiplication_program}")
assert compiled_addition == addition_program
assert compiled_multiplication == multiplication_program

Addition compilée : 2/3, 2/5
Multiplication compilée : 182/55, 17/11, 95/119, 1/17, 11/13, 17/19, 11/3, 1/5
Addition identique à la source : True
Multiplication identique à la source : True


### Lecture du résultat

La compilation donne exactement les programmes utilisés plus haut : `2/3, 2/5` pour l'addition, et les huit fractions `182/55, 17/11, 95/119, 1/17, 11/13, 17/19, 11/3, 1/5` pour la multiplication. Tester la divisibilité par le dénominateur, c'est tester si les registres du gardien peuvent être décrémentés ; multiplier par le numérateur, c'est appliquer les increments. La boucle de contrôle est portée par l'ordre des fractions.

## À retenir

| Idée | Forme FRACTRAN |
|---|---|
| État de la machine | un entier $n$ |
| Registres | exposants des nombres premiers dans $n$ |
| Test d'un registre | divisibilité de $n$ par le dénominateur |
| Incrément / décrément | multiplication par le numérateur / division par le dénominateur |
| Ordre du programme | contrôle de flux |

FRACTRAN montre que le calcul universel n'a besoin ni de mémoire adressable ni d'instructions conditionnelles explicites : une liste ordonnée de fractions et un entier suffisent.

## Vérifiez votre compréhension

1. Pourquoi la fraction $\frac{2}{3}$ décrémente-t-elle le registre associé à 3 ?
2. Quelle information de contrôle l'ordre des fractions encode-t-il ?
3. Pourquoi l'arrêt du programme Collatz pour tout $2^x$ équivaut-il à la conjecture de Collatz ?

<details>
<summary>Réponses</summary>

1. Diviser par 3 réduit l'exposant de 3 d'une unité ; multiplier par 2 augmente celui de 2 d'une unité.
2. La première fraction applicable détermine la règle qui s'exécute, donc l'ordre simule les branchements conditionnels.
3. Les états puissance de 2 reproduisent exactement la suite de Collatz ; si le programme s'arrête toujours sur $2^1$, la suite atteint toujours 1.

</details>

## Pour aller plus loin

- Le chapitre de Conway (1987) dans *Open Problems in Communication and Computation* présente aussi PRIMEGAME et POLYGAME.
- Comparer avec GOL-09 : Hashlife et FRACTRAN sont deux réponses de Conway à la même question, « quelle est la règle la plus simple qui calcule ? ».